## ACUTE MYELOID LUEKANEMIA PHENOTYPE FILE 

In [51]:
# Load packages 
from sklearn.preprocessing import StandardScaler
from functools import reduce
import pandas as pd 
import os 

In [52]:
# Workspace bucket 
my_bucket = os.getenv("WORKSPACE_BUCKET")

In [53]:
# Make directory for aml phenotypes files 
!rm -rf aml_phenotype_files
!mkdir -p aml_phenotype_files

In [54]:
# Assign sex variable 
sex['sex'] = sex['dragen_sex_ploidy'].apply(lambda x: 1 if x == "XX" else 0 if x == "XY" else 2)
sex['sex'] = sex['sex'].astype("int")

In [55]:
import pandas
import os

# This query represents dataset "aml_cohort" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_39009800_person_sql = """
    SELECT
        person.person_id 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person   
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (140352)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 )) criteria 
            UNION
            DISTINCT SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN (4304199) 
                    AND is_standard = 1 )) criteria ) )"""

aml_person_df = pandas.read_gbq(
    dataset_39009800_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

Downloading:   0%|          | 0/774 [00:00<?, ?rows/s]

In [56]:
import pandas
import os

# This query represents dataset "electronic_health_records" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_72216074_person_sql = """
    SELECT
        person.person_id 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person   
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) )"""

ehr_person_df = pandas.read_gbq(
    dataset_72216074_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

Downloading:   0%|          | 0/393601 [00:00<?, ?rows/s]

In [57]:
import pandas
import os

# This query represents dataset "zip_code_socioeconmic_cohort" for domain "zip_code_socioeconomic" and was generated for All of Us Controlled Tier Dataset v8
dataset_01113255_zip_code_socioeconomic_sql = """
    SELECT
        observation.person_id,
        observation.observation_datetime,
        zip_code.zip3_as_string as zip_code,
        zip_code.fraction_assisted_income as assisted_income,
        zip_code.fraction_high_school_edu as high_school_education,
        zip_code.median_income,
        zip_code.fraction_no_health_ins as no_health_insurance,
        zip_code.fraction_poverty as poverty,
        zip_code.fraction_vacant_housing as vacant_housing,
        zip_code.deprivation_index,
        zip_code.acs as american_community_survey_year 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.zip3_ses_map` zip_code 
    JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.observation` observation 
            ON CAST(SUBSTR(observation.value_as_string, 0, STRPOS(observation.value_as_string, '*') - 1) AS INT64) = zip_code.zip3 
            AND observation_source_concept_id = 1585250 
            AND observation.value_as_string NOT LIKE 'Res%'"""

zip_code_socioeconomic_df = pandas.read_gbq(
    dataset_01113255_zip_code_socioeconomic_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

zip_code_socioeconomic_df.head(5)

Downloading:   0%|          | 0/606414 [00:00<?, ?rows/s]

,person_id,observation_datetime,zip_code,assisted_income,high_school_education,median_income,no_health_insurance,poverty,vacant_housing,deprivation_index,american_community_survey_year
0,2384063,2019-08-24 18:41:16+00:00,916**,9.498458,82.377652,61386.266101,14.349386,16.722471,5.277623,0.341010,2017
1,2055202,2020-04-12 20:41:48+00:00,236**,12.356209,90.734513,59619.001555,10.297056,13.807889,9.947325,0.311358,2017
2,6787387,2023-05-12 17:13:51+00:00,954**,10.252124,86.063087,64747.379864,9.642902,13.514230,11.035914,0.312343,2017
3,6621807,2023-06-30 01:43:28+00:00,009**,28.088321,81.240046,29298.309620,8.067842,34.802099,18.547342,0.460629,2017
4,1073226,2019-01-16 21:15:05+00:00,107**,16.108189,83.585482,76301.804225,8.968810,14.613248,8.886805,0.320318,2017


In [58]:
# Combine list of quality control information and demographic information 
phenotype_file = zip_code_socioeconomic_df[zip_code_socioeconomic_df["person_id"].isin(ehr_person_df["person_id"])]

In [59]:
# Assign AML phenotype to individuals 
phenotype_file['aml_phenotype'] = phenotype_file['person_id'].isin(aml_person_df['person_id']).astype(int)

/tmp/ipykernel_121/4252290690.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  phenotype_file['aml_phenotype'] = phenotype_file['person_id'].isin(aml_person_df['person_id']).astype(int)


In [60]:
# Format phenotype file  
phenotype_file = (phenotype_file[['aml_phenotype',
            'deprivation_index', 'assisted_income', 
            'high_school_education','median_income',
            'no_health_insurance', 'poverty',
            'vacant_housing', 
            'person_id']]
)

In [61]:
# Identify columns that start with "PC" or "age"
cols_to_standardize = [col for col in phenotype_file.columns if col.startswith("PC") 
                       or col.startswith("age") 
                       or col.startswith("deprivation_index")
                       or col.startswith("assisted_income")
                       or col.startswith("high_school_education")
                       or col.startswith("median_income")
                       or col.startswith("no_health_insurance")
                       or col.startswith("poverty")
                       or col.startswith("vacant_housing")]
# Standardize only the selected columns
scaler = StandardScaler()
phenotype_file[cols_to_standardize] = scaler.fit_transform(phenotype_file[cols_to_standardize])
# Print results of the standarization 
print("============================================")
print("Check variances of standardized covariates:")
print("============================================")
print(phenotype_file.filter(regex="PC|age").var())
print(phenotype_file.describe())
print("============================================")

Check variances of standardized covariates:
Series([], dtype: float64)
       aml_phenotype  deprivation_index  assisted_income  \
count  376904.000000       3.769040e+05     3.769040e+05   
mean        0.001953      -2.158187e-16     8.299443e-16   
std         0.044147       1.000001e+00     1.000001e+00   
min         0.000000      -2.776421e+00    -2.004859e+00   
25%         0.000000      -6.338469e-01    -6.609280e-01   
50%         0.000000      -1.349089e-01    -1.872823e-01   
75%         0.000000       6.096715e-01     3.858298e-01   
max         1.000000       4.854855e+00     4.790400e+00   

       high_school_education  median_income  no_health_insurance  \
count           3.769040e+05   3.769040e+05         3.769040e+05   
mean           -1.013149e-15   8.663288e-16         7.510672e-17   
std             1.000001e+00   1.000001e+00         1.000001e+00   
min            -4.382383e+00  -2.912135e+00        -1.929892e+00   
25%            -5.856505e-01  -6.248443e-01     

In [62]:
# Final check for phenotype 
print("============================================")
print("Check the presence of aml phenotype:")
print("============================================")
print(phenotype_file["aml_phenotype"].value_counts())
print("============================================")

Check the presence of aml phenotype:
aml_phenotype
0    376168
1       736
Name: count, dtype: int64


In [63]:
# Save phenoytpe file
phenotype_file.to_csv("aml_phenotype_deprivation_index.txt", sep ='\t', index=None)